# Day 56 — Orchestration basics: Prefect intro
Objectives:
- Create simple Prefect flows and tasks.
- Parameterize a pipeline (preprocess → train → evaluate).
- Run locally and observe logs.
Note: `pip install prefect` (already added to requirements.txt).


In [ ]:
from prefect import flow, task

@task
def load_data():
    import seaborn as sns
    df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','fare','age'])
    return df

@task
def preprocess(df):
    import pandas as pd
    df = df.copy()
    df['fare'] = pd.to_numeric(df['fare'], errors='coerce').fillna(df['fare'].median())
    df = df.dropna(subset=['age'])
    return df

@task
def train(df):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from pathlib import Path

    import joblib
    X = df[['sex','class','fare','age']]; y = df['survived']
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe.fit(Xtr,ytr)
    import numpy as np
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    artifact_dir = Path('artifacts/day56')
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, artifact_dir / 'prefect_titanic_pipeline.joblib')
    return auc

@flow
def training_flow():
    df = load_data()
    df2 = preprocess(df)
    auc = train(df2)
    print('AUC:', auc)

# Run the flow
if __name__ == '__main__':
    training_flow()


## Exercises
1) Add parameters (e.g., test_size, random_state) to the flow.
2) Split the training task into train and evaluate tasks with explicit outputs.
3) Explore Prefect UI (optional) and scheduling basics.